In [1]:
import os
import glob
import pickle
import numpy as np

SOURCE = 'labeled_data/MECO/video'

# ---------- 1. Find pickle files ----------
if os.path.isfile(SOURCE):
    files = [SOURCE]
else:
    files = sorted(
        glob.glob(os.path.join(SOURCE, '**', '*.pkl'), recursive=True)
        + glob.glob(os.path.join(SOURCE, '**', '*.pickle'), recursive=True)
        + glob.glob(os.path.join(SOURCE, '**', '*.pckl'), recursive=True)
    )

print(f"Found {len(files)} pickle file(s):")
for f in files[:42]:
    print("  ", f, f"({os.path.getsize(f)/1e6:.2f} MB)")
if not files:
    raise SystemExit("No pickle files found, check SOURCE path.")


# ---------- 2. Loader (falls back to pandas / latin1 for old py2 pickles) ----------
def load_pickle(path):
    try:
        with open(path, 'rb') as fh:
            return pickle.load(fh)
    except Exception as e1:
        try:
            import pandas as pd
            return pd.read_pickle(path)
        except Exception as e2:
            with open(path, 'rb') as fh:
                return pickle.load(fh, encoding='latin1')


# ---------- 3. Recursive inspector ----------
def describe(obj, name='root', depth=0, max_depth=4, max_items=15):
    pad = '  ' * depth
    t = type(obj).__name__

    try:
        import pandas as pd
    except ImportError:
        pd = None

    if pd is not None and isinstance(obj, pd.DataFrame):
        print(f"{pad}{name}: DataFrame shape={obj.shape}")
        print(f"{pad}  index: {obj.index.dtype}, {obj.index[:3].tolist()} ...")
        print(f"{pad}  columns ({len(obj.columns)}): {list(obj.columns)}")
        print(f"{pad}  dtypes:\n{obj.dtypes.value_counts().to_string()}")
        print(f"{pad}  head:\n{obj.head().to_string()}")
        au_cols = [c for c in obj.columns if 'au' in str(c).lower()]
        if au_cols:
            print(f"{pad}  AU-like columns ({len(au_cols)}): {au_cols}")
        print(f"{pad}  NaNs per column (non-zero only):")
        nans = obj.isna().sum()
        print(nans[nans > 0].to_string() if (nans > 0).any() else f"{pad}    none")
        print(f"{pad}  describe:\n{obj.describe().T.head(30).to_string()}")

    elif pd is not None and isinstance(obj, pd.Series):
        print(f"{pad}{name}: Series len={len(obj)} dtype={obj.dtype}")
        print(obj.head().to_string())

    elif isinstance(obj, np.ndarray):
        print(f"{pad}{name}: ndarray shape={obj.shape} dtype={obj.dtype}")
        if obj.size and np.issubdtype(obj.dtype, np.number):
            print(f"{pad}  min={np.nanmin(obj):.4g} max={np.nanmax(obj):.4g} "
                  f"mean={np.nanmean(obj):.4g} nans={np.isnan(obj).sum()}")

    elif isinstance(obj, dict):
        print(f"{pad}{name}: dict with {len(obj)} keys")
        keys = list(obj.keys())
        print(f"{pad}  keys (first {max_items}): {keys[:max_items]}")
        if depth < max_depth:
            for k in keys[:max_items]:
                describe(obj[k], name=f"[{k!r}]", depth=depth + 1,
                         max_depth=max_depth, max_items=max_items)

    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{name}: {t} len={len(obj)}")
        if depth < max_depth:
            for i, v in enumerate(obj[:3]):
                describe(v, name=f"[{i}]", depth=depth + 1,
                         max_depth=max_depth, max_items=max_items)

    else:
        r = repr(obj)
        print(f"{pad}{name}: {t} -> {r[:100]}{'...' if len(r) > 100 else ''}")


# ---------- 4. Run on each file ----------
# for path in files[0:1]:   # change/remove the slice to inspect more
path = files[0]
print("\n" + "=" * 80)
print("FILE:", path)
print("=" * 80)
data = load_pickle(path)
describe(data)

Found 42 pickle file(s):
   labeled_data/MECO/video/video_S1.pkl (28.18 MB)
   labeled_data/MECO/video/video_S10.pkl (27.56 MB)
   labeled_data/MECO/video/video_S11.pkl (27.56 MB)
   labeled_data/MECO/video/video_S12.pkl (27.37 MB)
   labeled_data/MECO/video/video_S13.pkl (24.07 MB)
   labeled_data/MECO/video/video_S14.pkl (23.23 MB)
   labeled_data/MECO/video/video_S15.pkl (27.41 MB)
   labeled_data/MECO/video/video_S16.pkl (27.56 MB)
   labeled_data/MECO/video/video_S17.pkl (24.29 MB)
   labeled_data/MECO/video/video_S18.pkl (27.67 MB)
   labeled_data/MECO/video/video_S19.pkl (27.63 MB)
   labeled_data/MECO/video/video_S2.pkl (27.63 MB)
   labeled_data/MECO/video/video_S20.pkl (21.32 MB)
   labeled_data/MECO/video/video_S21.pkl (27.63 MB)
   labeled_data/MECO/video/video_S22.pkl (27.74 MB)
   labeled_data/MECO/video/video_S23.pkl (27.92 MB)
   labeled_data/MECO/video/video_S24.pkl (27.63 MB)
   labeled_data/MECO/video/video_S25.pkl (27.56 MB)
   labeled_data/MECO/video/video_S26.pkl 

In [2]:
print(data['emotion_to_idx_sa']) # Can chose which session to inspect, e.g. session1, session2, session3 or emotion_to_idx_sa
print('--------------------------------')

s1 = data['session1']
s1_keys = list(s1.keys())
print(s1_keys)
print(s1)

{'angry': 0, 'boredom': 1, 'sadness': 2, 'neutral': 3, 'tension': 4, 'happiness': 5}
--------------------------------
['deep_feature', 'au_openface', 'hp_openface', 'gz_openface', 'label_vi_class', 'label_valence', 'label_arousal', 'emotion_to_idx_vi', 'label_sa_class']
{'deep_feature': [tensor([[[ 0.8795, -1.7685,  0.8400,  ..., -1.0537, -2.1933,  0.5585],
         [ 0.8793, -1.7690,  0.8399,  ..., -1.0535, -2.1938,  0.5584],
         [ 0.8799, -1.7631,  0.8455,  ..., -1.0576, -2.1933,  0.5569],
         ...,
         [ 0.8938, -1.7741,  0.8422,  ..., -1.0549, -2.2036,  0.5504],
         [ 0.8898, -1.7711,  0.8414,  ..., -1.0528, -2.1993,  0.5537],
         [ 0.8900, -1.7696,  0.8416,  ..., -1.0535, -2.1985,  0.5537]]]), tensor([[[ 0.8898, -1.7711,  0.8414,  ..., -1.0528, -2.1993,  0.5537],
         [ 0.8900, -1.7696,  0.8416,  ..., -1.0535, -2.1985,  0.5537],
         [ 0.8783, -1.7751,  0.8466,  ..., -1.0514, -2.2040,  0.5553],
         ...,
         [ 0.8797, -1.7731,  0.8439,  ...

In [3]:
au = s1['au_openface']
# au_keys = list(s1.keys())
# print(s1_keys)
print('Length of action units:', len(au))
print(au)


Length of action units: 247
[tensor([[0.0000, 0.0000, 1.3600, 0.0000, 2.5800, 2.6500, 1.1000, 2.4000, 0.8600,
         0.9100, 1.3900, 2.6500, 0.6800, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 1.0000, 0.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 0.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.9900, 0.0000, 2.1600, 2.1900, 0.8200, 1.9300, 0.6800,
         0.7300, 0.9500, 2.2500, 0.3600, 0.0800, 0.0800, 0.0000, 0.0800, 0.0000,
         0.0000, 1.0000, 0.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 1.0000,
         1.0000, 1.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5400, 0.0500, 1.7500, 1.8500, 0.4600, 1.5000, 0.5300,
         0.4300, 0.6700, 1.8900, 0.1400, 0.0800, 0.0800, 0.1200, 0.1500, 0.0000,
         1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 1.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0

In [4]:
print(au[0].shape)

torch.Size([16, 35])


In [5]:
labels_vi = s1['label_vi_class']
labels_sa = s1['label_sa_class']
emotions_to_idx = s1['emotion_to_idx_vi']

print(labels_vi)
print(labels_sa)
print(emotions_to_idx)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [6]:
"""
Loading + preprocessing for MECO window-level features.

Assumed layout per subject dict `d` (adapt KEYS to what `describe(d)` prints):
    d[<au key>] : (N, 16, 35)   16 frames x 35 OpenFace AUs per 4 s window
    d[<hp key>] : (N, 16, 6)    head pose (optional)
    d[<eg key>] : (N, 16, 2)    gaze angles (optional)
    d['label_vi_class'], d['label_sa_class'] : (N,)
"""
import numpy as np
import torch
from torch.utils.data import Dataset


def describe(d):
    """Print every key with its type/shape so you can fill in KEYS."""
    for k, v in d.items():
        shape = getattr(v, "shape", None)
        if shape is None and isinstance(v, (list, tuple)):
            shape = ("list", len(v))
        print(f"{k:30s} {type(v).__name__:12s} {shape}")


def trial_ids_from_labels(y_vi):
    """Consecutive windows with the same stimulus label = one trial.
    (Each session has one trial per emotion, so neighbouring trials differ.)"""
    y = np.asarray(y_vi)
    return np.concatenate([[0], np.cumsum(y[1:] != y[:-1])])


def load_subject(d, subject_id, keys, label_key="label_vi_class", label_map=None):
    """keys: {'au': '<key in d>', 'hp': '<key>', 'eg': '<key>'} (hp/eg optional)."""
    out = {m: np.asarray(d[k], dtype=np.float32) for m, k in keys.items()}
    y = np.asarray(d[label_key]).astype(int)
    if label_map is not None:  # e.g. 6-class self-report -> 5-class (boredom into neutral)
        y = np.array([label_map[v] for v in y])
    for m, a in out.items():
        assert a.ndim == 3 and a.shape[0] == len(y), f"{m}: expected (N,T,F), got {a.shape}"
    out["y"] = y
    out["subject"] = np.full(len(y), subject_id)
    out["trial"] = trial_ids_from_labels(d["label_vi_class"])
    return out


def concat_subjects(parts):
    """Stack per-subject dicts; make trial ids globally unique."""
    offset, fixed = 0, []
    for p in parts:
        p = dict(p)
        p["trial"] = p["trial"] + offset
        offset = p["trial"].max() + 1
        fixed.append(p)
    return {k: np.concatenate([p[k] for p in fixed]) for k in fixed[0]}


def normalize_per_subject(data, mods=("au", "hp", "eg")):
    """z-score each feature per subject. Binary columns (OpenFace AU_c presence
    flags) are left untouched. Uses only the subject's own statistics."""
    for s in np.unique(data["subject"]):
        idx = np.where(data["subject"] == s)[0]
        for m in mods:
            if m not in data:
                continue
            a = data[m][idx]
            flat = a.reshape(-1, a.shape[-1])
            mu, sd = np.nanmean(flat, 0), np.nanstd(flat, 0) + 1e-6
            binary = np.array([np.isin(np.unique(c[~np.isnan(c)]), [0, 1]).all() for c in flat.T])
            mu[binary], sd[binary] = 0.0, 1.0
            data[m][idx] = (a - mu) / sd
    return data


class MecoDataset(Dataset):
    """Returns ({mod: (T,F)}, missing_mask (T,), label).
    NaN frames (failed face detection) are zero-filled and flagged in the mask."""

    def __init__(self, data, idx, mods=("au", "hp", "eg"), train=False,
                 noise=0.05, frame_drop=0.1):
        self.data, self.idx, self.mods = data, np.asarray(idx), [m for m in mods if m in data]
        self.train, self.noise, self.frame_drop = train, noise, frame_drop

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        x = {m: torch.from_numpy(self.data[m][j]) for m in self.mods}
        T = next(iter(x.values())).shape[0]
        miss = torch.zeros(T, dtype=torch.bool)
        for m in self.mods:
            miss |= torch.isnan(x[m]).any(-1)
            x[m] = torch.nan_to_num(x[m])
        if self.train:
            for m in self.mods:
                x[m] = x[m] + self.noise * torch.randn_like(x[m])
            miss |= torch.rand(T) < self.frame_drop
            if miss.all():
                miss[torch.randint(T, (1,))] = False
        return x, miss, int(self.data["y"][j])


def make_synthetic(n_subjects=8, n_trials=6, wins_per_trial=20, T=16, n_classes=6, seed=0):
    """Random data with the same layout, for smoke-testing the pipeline."""
    rng = np.random.default_rng(seed)
    parts = []
    for s in range(n_subjects):
        y = np.repeat(rng.permutation(n_classes)[:n_trials], wins_per_trial)
        au = rng.normal(size=(len(y), T, 35)).astype(np.float32)
        au[..., 17:] = (rng.random((len(y), T, 18)) > 0.7).astype(np.float32)  # binary AU_c
        au[..., 0] += y[:, None] * 0.5                                         # weak signal
        d = {"au": au, "hp": rng.normal(size=(len(y), T, 6)).astype(np.float32),
             "eg": rng.normal(size=(len(y), T, 2)).astype(np.float32),
             "label_vi_class": y, "label_sa_class": y}
        parts.append(load_subject(d, s, {"au": "au", "hp": "hp", "eg": "eg"}))
    return concat_subjects(parts)

# Build dataset

In [28]:
import os
import glob
import pickle
import numpy as np

SOURCE = 'labeled_data/MECO/video'

# ---------- 1. Find pickle files ----------
if os.path.isfile(SOURCE):
    files = [SOURCE]
else:
    files = sorted(
        glob.glob(os.path.join(SOURCE, '**', '*.pkl'), recursive=True)
        + glob.glob(os.path.join(SOURCE, '**', '*.pickle'), recursive=True)
        + glob.glob(os.path.join(SOURCE, '**', '*.pckl'), recursive=True)
    )

print(f"Found {len(files)} pickle file(s):")
# for f in files[:42]:
#     print("  ", f, f"({os.path.getsize(f)/1e6:.2f} MB)")
if not files:
    raise SystemExit("No pickle files found, check SOURCE path.")


# ---------- 2. Loader (falls back to pandas / latin1 for old py2 pickles) ----------
def load_pickle(path):
    try:
        with open(path, 'rb') as fh:
            return pickle.load(fh)
    except Exception as e1:
        try:
            import pandas as pd
            return pd.read_pickle(path)
        except Exception as e2:
            with open(path, 'rb') as fh:
                return pickle.load(fh, encoding='latin1')


# ---------- 3. Recursive inspector ----------
def describe(obj, name='root', depth=0, max_depth=4, max_items=15):
    pad = '  ' * depth
    t = type(obj).__name__

    try:
        import pandas as pd
    except ImportError:
        pd = None

    if pd is not None and isinstance(obj, pd.DataFrame):
        print(f"{pad}{name}: DataFrame shape={obj.shape}")
        print(f"{pad}  index: {obj.index.dtype}, {obj.index[:3].tolist()} ...")
        print(f"{pad}  columns ({len(obj.columns)}): {list(obj.columns)}")
        print(f"{pad}  dtypes:\n{obj.dtypes.value_counts().to_string()}")
        print(f"{pad}  head:\n{obj.head().to_string()}")
        au_cols = [c for c in obj.columns if 'au' in str(c).lower()]
        if au_cols:
            print(f"{pad}  AU-like columns ({len(au_cols)}): {au_cols}")
        print(f"{pad}  NaNs per column (non-zero only):")
        nans = obj.isna().sum()
        print(nans[nans > 0].to_string() if (nans > 0).any() else f"{pad}    none")
        print(f"{pad}  describe:\n{obj.describe().T.head(30).to_string()}")

    elif pd is not None and isinstance(obj, pd.Series):
        print(f"{pad}{name}: Series len={len(obj)} dtype={obj.dtype}")
        print(obj.head().to_string())

    elif isinstance(obj, np.ndarray):
        print(f"{pad}{name}: ndarray shape={obj.shape} dtype={obj.dtype}")
        if obj.size and np.issubdtype(obj.dtype, np.number):
            print(f"{pad}  min={np.nanmin(obj):.4g} max={np.nanmax(obj):.4g} "
                  f"mean={np.nanmean(obj):.4g} nans={np.isnan(obj).sum()}")

    elif isinstance(obj, dict):
        print(f"{pad}{name}: dict with {len(obj)} keys")
        keys = list(obj.keys())
        print(f"{pad}  keys (first {max_items}): {keys[:max_items]}")
        if depth < max_depth:
            for k in keys[:max_items]:
                describe(obj[k], name=f"[{k!r}]", depth=depth + 1,
                         max_depth=max_depth, max_items=max_items)

    elif isinstance(obj, (list, tuple)):
        print(f"{pad}{name}: {t} len={len(obj)}")
        if depth < max_depth:
            for i, v in enumerate(obj[:3]):
                describe(v, name=f"[{i}]", depth=depth + 1,
                         max_depth=max_depth, max_items=max_items)

    else:
        r = repr(obj)
        print(f"{pad}{name}: {t} -> {r[:100]}{'...' if len(r) > 100 else ''}")

X = np.zeros((30592, 1, 16, 35)) # Change 1 to 3 if I should include head positon and gaze too
Y = np.zeros((30592, 1))
# ---------- 4. Run on each file ----------
for path in files:   # change/remove the slice to inspect more
    # path = files[0]
    print("\n" + "=" * 80)
    print("FILE:", path)
    print("=" * 80)
    data = load_pickle(path)
    labels = data['session1']['label_vi_class']
    print('Length of labels:', len(labels))
    Y[:len(labels)] = np.array(labels).reshape(-1, 1)
    X[:len(labels), 0] = data['session1']['au_openface']
    # X[:len(labels), 1] = data['session1']['hp_openface']
    # X[:len(labels), 2] = data['session1']['gz_openface']

    labels = data['session2']['label_vi_class']
    print('Length of labels:', len(labels))
    Y[len(labels):2*len(labels)] = np.array(labels).reshape(-1, 1)
    X[len(labels):2*len(labels), 0] = data['session2']['au_openface']
    # X[:2*len(labels), 1] = data['session2']['hp_openface']
    # X[:2*len(labels), 2] = data['session2']['gz_openface']

    labels = data['session3']['label_vi_class']
    print('Length of labels:', len(labels))
    Y[2*len(labels):3*len(labels)] = np.array(labels).reshape(-1, 1)
    X[2*len(labels):3*len(labels), 0] = data['session3']['au_openface']
    # X[:3*len(labels), 1] = data['session3']['hp_openface']
    # X[:3*len(labels), 2] = data['session3']['gz_openface']
    
#describe(data)

print(X)




    

Found 42 pickle file(s):

FILE: labeled_data/MECO/video/video_S1.pkl
Length of labels: 247
Length of labels: 249
Length of labels: 272

FILE: labeled_data/MECO/video/video_S10.pkl
Length of labels: 248
Length of labels: 245
Length of labels: 258

FILE: labeled_data/MECO/video/video_S11.pkl
Length of labels: 250
Length of labels: 244
Length of labels: 257

FILE: labeled_data/MECO/video/video_S12.pkl
Length of labels: 247
Length of labels: 243
Length of labels: 256

FILE: labeled_data/MECO/video/video_S13.pkl
Length of labels: 248
Length of labels: 245
Length of labels: 163

FILE: labeled_data/MECO/video/video_S14.pkl
Length of labels: 248
Length of labels: 248
Length of labels: 137

FILE: labeled_data/MECO/video/video_S15.pkl
Length of labels: 247
Length of labels: 244
Length of labels: 256

FILE: labeled_data/MECO/video/video_S16.pkl
Length of labels: 248
Length of labels: 244
Length of labels: 259

FILE: labeled_data/MECO/video/video_S17.pkl
Length of labels: 248
Length of labels: 245

In [30]:
"""
Build the dataset for s2d_au.train_cv from the MECO pickles.

Layout (from your inspection): one pickle per subject, each a dict
    {'session1': {...}, 'session2': {...}, 'session3': {...}}
and every session dict holds
    'au_openface' (n,16,35), 'hp_openface', 'gz_openface', 'label_vi_class', 'label_sa_class'
Sessions are appended one after another (they may have different n).
"""
import glob
import os
import pickle
import re

import numpy as np

# from meco_data import concat_subjects, trial_ids_from_labels

# ------------------------------------------------------------------ EDIT IF NEEDED
SOURCE = "labeled_data/MECO/video"
MODS = {"au": "au_openface", "hp": "hp_openface", "eg": "gz_openface"}  # our name -> pickle key
LABEL_VI, LABEL_SA = "label_vi_class", "label_sa_class"


def subject_from_path(path):
    """One file = one subject, so the file name is the subject id.
    CHECK the printed mapping; change this if a subject spans several files."""
    return os.path.splitext(os.path.basename(path))[0]
# --------------------------------------------------------------------------------


def find_files(source=SOURCE):
    if os.path.isfile(source):
        return [source]
    pats = ("*.pkl", "*.pickle", "*.pckl")
    return sorted(f for p in pats for f in glob.glob(os.path.join(source, "**", p), recursive=True))


def load_pickle(path):
    try:
        with open(path, "rb") as fh:
            return pickle.load(fh)
    except Exception:
        try:
            import pandas as pd
            return pd.read_pickle(path)
        except Exception:
            with open(path, "rb") as fh:
                return pickle.load(fh, encoding="latin1")


def build_dataset(files=None, mods=("au",), label="vi", verbose=True):
    """mods: subset of MODS keys to load, e.g. ('au',) or ('au','hp','eg').
    label: 'vi' (stimulus, no mapping needed) or 'sa' (raw 6-class self-report)."""
    files = files or find_files()
    parts = []
    for fi, path in enumerate(files):
        raw = load_pickle(path)
        sid = subject_from_path(path)
        sessions = sorted(k for k in raw if re.fullmatch(r"session\d+", str(k)))
        if not sessions:
            raise KeyError(f"{path}: no 'sessionN' keys, found {list(raw)[:10]}")
        counts = []
        for sess in sessions:
            s = raw[sess]
            y_vi = np.asarray(s[LABEL_VI]).astype(int).ravel()
            y = y_vi if label == "vi" else np.asarray(s[LABEL_SA]).astype(int).ravel()
            part = {}
            for m in mods:
                a = np.asarray(s[MODS[m]], dtype=np.float32)
                assert a.ndim == 3 and a.shape[0] == len(y_vi), \
                    f"{path}/{sess}/{MODS[m]}: expected (n,16,F) with n={len(y_vi)}, got {a.shape}"
                part[m] = a
            part.update(y=y, y_vi=y_vi,
                        subject=np.full(len(y), sid),
                        session=np.full(len(y), int(re.search(r"\d+", sess).group())),
                        trial=trial_ids_from_labels(y_vi))
            parts.append(part)
            counts.append(len(y))
        if verbose:
            if fi == 0:
                print("keys in first session dict:", list(raw[sessions[0]].keys()))
            print(f"{os.path.basename(path):30s} -> subject '{sid}' | windows per session {counts}")
    data = concat_subjects(parts)
    if verbose:
        print(f"\nfiles: {len(files)} | subjects: {len(np.unique(data['subject']))} "
              f"| windows: {len(data['y'])} (paper: 30592) | trials: {len(np.unique(data['trial']))}")
        print("feature dims:", {m: data[m].shape[-1] for m in mods})
        print("class counts:", np.bincount(data["y"]))
        print("NaN fraction:", {m: float(np.isnan(data[m]).mean()) for m in mods})
    return data

In [39]:
"""
Simple MLP baseline for MECO window features.

  (N,16,F) frames  ->  per-window stats (mean, std, min, max, mean |delta|)  ->  MLP  ->  classes

Expects `data` from build_dataset(...) after normalize_per_subject(...):
    data[mod]: (N,16,F), data['y'], data['subject'], data['trial']
"""
import copy
import warnings

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import GroupKFold


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():  # Apple-silicon GPU
        return "mps"
    return "cpu"


def window_features(data, mods=("au",)):
    """(N,T,F) per modality -> (N, 5*F) stats per modality, concatenated.
    NaN frames (failed detections) are ignored; an all-NaN window becomes zeros."""
    feats = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        for m in mods:
            a = data[m]
            feats += [np.nanmean(a, 1), np.nanstd(a, 1), np.nanmin(a, 1), np.nanmax(a, 1),
                      np.nanmean(np.abs(np.diff(a, axis=1)), 1)]
    return np.nan_to_num(np.concatenate(feats, 1)).astype(np.float32)


class SimpleMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=128, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden // 2, n_classes))

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def _evaluate(model, X, y, crit, chunk=4096):
    model.eval()
    logits = torch.cat([model(X[i:i + chunk]) for i in range(0, len(X), chunk)])
    p, yy = logits.argmax(-1).cpu().numpy(), y.cpu().numpy()
    return {"loss": crit(logits, y).item(),
            "UAR": recall_score(yy, p, average="macro", zero_division=0),
            "WAR": accuracy_score(yy, p)}


def _shuffle_trial_labels(y, trial, rng):
    """Control: permute labels across trials (labels stay constant within a trial)."""
    trials = np.unique(trial)
    lab = np.array([y[trial == t][0] for t in trials])
    mapping = dict(zip(trials, rng.permutation(lab)))
    return np.array([mapping[t] for t in trial])


def train_cv_mlp(data, n_classes, mods=("au",), folds=5, epochs=60, bs=128, lr=1e-3, wd=1e-2,
                 hidden=128, drop=0.3, noise=0.05, seed=0, dev=None,
                 shuffle_labels=False, print_every=1):
    dev = dev or get_device()
    X_all, y_all = window_features(data, mods), np.asarray(data["y"])
    subj, trial = np.asarray(data["subject"]), np.asarray(data["trial"])
    rng = np.random.default_rng(seed)
    print(f"device: {dev} | features: {X_all.shape} | subjects: {len(np.unique(subj))} "
          f"| windows: {len(y_all)} | trials: {len(np.unique(trial))}")
    if shuffle_labels:
        print("!! CONTROL RUN: labels shuffled across trials, expect chance-level scores")
        y_all = _shuffle_trial_labels(y_all, trial, rng)

    results, baselines = [], []
    for f, (tr, te) in enumerate(GroupKFold(folds).split(X_all, groups=subj)):
        tr_subj = rng.permutation(np.unique(subj[tr]))
        val_subj = tr_subj[: max(1, int(0.15 * len(tr_subj)))]
        is_val = np.isin(subj[tr], val_subj)
        va, tr = tr[is_val], tr[~is_val]

        for a, b, name in [(tr, te, "train/test"), (tr, va, "train/val"), (va, te, "val/test")]:
            assert not set(subj[a]) & set(subj[b]), f"subject overlap {name}"
            assert not set(trial[a]) & set(trial[b]), f"trial overlap {name}"
        base = (y_all[te] == np.bincount(y_all[tr], minlength=n_classes).argmax()).mean()
        baselines.append(base)
        print(f"\n=== fold {f} | train {len(tr)} ({len(set(subj[tr]))} subj) | val {len(va)} | "
              f"test {len(te)} ({len(set(subj[te]))} subj) | majority-class WAR on test: {base:.3f}")

        # standardise with TRAIN statistics only
        mu, sd = X_all[tr].mean(0), X_all[tr].std(0) + 1e-6
        T = lambda idx: torch.from_numpy((X_all[idx] - mu) / sd).to(dev)
        Y = lambda idx: torch.from_numpy(y_all[idx]).long().to(dev)
        Xtr, Xva, Xte, ytr, yva, yte = T(tr), T(va), T(te), Y(tr), Y(va), Y(te)

        torch.manual_seed(seed + f)
        model = SimpleMLP(X_all.shape[1], n_classes, hidden, drop).to(dev)
        counts = np.bincount(y_all[tr], minlength=n_classes).astype(np.float32)
        w = torch.tensor(counts.sum() / (n_classes * np.maximum(counts, 1)),
                         dtype=torch.float32, device=dev)
        crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

        best, best_state, best_ep = -1, None, 0
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(len(Xtr), device=dev)
            tl, tc = 0.0, 0
            for i in range(0, len(perm), bs):
                idx = perm[i:i + bs]
                xb, yb = Xtr[idx], ytr[idx]
                if noise > 0:
                    xb = xb + noise * torch.randn_like(xb)
                logits = model(xb)
                loss = crit(logits, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
                tl += loss.item() * len(idx)
                tc += (logits.argmax(-1) == yb).sum().item()
            sched.step()
            v = _evaluate(model, Xva, yva, crit)
            mark = ""
            if v["UAR"] > best:
                best, best_ep, best_state = v["UAR"], ep + 1, copy.deepcopy(model.state_dict())
                mark = " *"
            if (ep + 1) % print_every == 0 or mark:
                print(f"fold {f} ep {ep + 1:02d}/{epochs} | train loss {tl / len(Xtr):.3f} "
                      f"acc {tc / len(Xtr):.3f} | val loss {v['loss']:.3f} "
                      f"UAR {v['UAR']:.3f} WAR {v['WAR']:.3f}{mark}")
        model.load_state_dict(best_state)
        r = _evaluate(model, Xte, yte, crit)
        print(f"fold {f} TEST (best val epoch {best_ep}): UAR {r['UAR']:.3f} WAR {r['WAR']:.3f} "
              f"(majority {base:.3f})")
        results.append(r)
    print("\nmean test UAR %.3f | WAR %.3f | majority-class WAR %.3f" % (
        np.mean([r["UAR"] for r in results]), np.mean([r["WAR"] for r in results]),
        np.mean(baselines)))
    return results


# Usage:
#   from load_meco import build_dataset
#   from meco_data import normalize_per_subject
#   data = normalize_per_subject(build_dataset(mods=("au",), label="vi"), mods=("au",))
#   train_cv_mlp(data, n_classes=6, mods=("au",))
#   train_cv_mlp(data, n_classes=6, mods=("au",), shuffle_labels=True)   # leakage control

In [35]:
data = build_dataset(mods=("au",), label="vi")   # prints the per-file summary
assert len(set(data["subject"])) == 42
data = normalize_per_subject(data, mods=("au",))


keys in first session dict: ['deep_feature', 'au_openface', 'hp_openface', 'gz_openface', 'label_vi_class', 'label_valence', 'label_arousal', 'emotion_to_idx_vi', 'label_sa_class']
video_S1.pkl                   -> subject 'video_S1' | windows per session [247, 249, 272]
video_S10.pkl                  -> subject 'video_S10' | windows per session [248, 245, 258]
video_S11.pkl                  -> subject 'video_S11' | windows per session [250, 244, 257]
video_S12.pkl                  -> subject 'video_S12' | windows per session [247, 243, 256]
video_S13.pkl                  -> subject 'video_S13' | windows per session [248, 245, 163]
video_S14.pkl                  -> subject 'video_S14' | windows per session [248, 248, 137]
video_S15.pkl                  -> subject 'video_S15' | windows per session [247, 244, 256]
video_S16.pkl                  -> subject 'video_S16' | windows per session [248, 244, 259]
video_S17.pkl                  -> subject 'video_S17' | windows per session [248, 24

In [38]:
data['trial']

array([  0,   0,   0, ..., 732, 732, 732], shape=(30592,))

In [41]:

train_cv_mlp(data, n_classes=6, epochs=30, mods=("au",))

device: mps | features: (30592, 175) | subjects: 42 | windows: 30592 | trials: 733

=== fold 0 | train 21304 (29 subj) | val 3392 | test 5896 (8 subj) | majority-class WAR on test: 0.229
fold 0 ep 01/30 | train loss 1.780 acc 0.226 | val loss 7.794 UAR 0.217 WAR 0.232 *
fold 0 ep 02/30 | train loss 1.734 acc 0.271 | val loss 5.019 UAR 0.219 WAR 0.227 *
fold 0 ep 03/30 | train loss 1.694 acc 0.309 | val loss 2.383 UAR 0.192 WAR 0.201
fold 0 ep 04/30 | train loss 1.659 acc 0.331 | val loss 9.649 UAR 0.189 WAR 0.200
fold 0 ep 05/30 | train loss 1.628 acc 0.355 | val loss 10.142 UAR 0.194 WAR 0.207
fold 0 ep 06/30 | train loss 1.602 acc 0.372 | val loss 10.117 UAR 0.197 WAR 0.213
fold 0 ep 07/30 | train loss 1.579 acc 0.389 | val loss 9.984 UAR 0.206 WAR 0.220
fold 0 ep 08/30 | train loss 1.558 acc 0.404 | val loss 10.768 UAR 0.208 WAR 0.222
fold 0 ep 09/30 | train loss 1.542 acc 0.409 | val loss 19.450 UAR 0.216 WAR 0.231
fold 0 ep 10/30 | train loss 1.528 acc 0.422 | val loss 22.355 UAR 

[{'loss': 6.330204963684082,
  'UAR': 0.17537706985080617,
  'WAR': 0.17757801899592945},
 {'loss': 4.945133686065674,
  'UAR': 0.1799692718805478,
  'WAR': 0.1830962059620596},
 {'loss': 1.9544895887374878,
  'UAR': 0.18908233452188872,
  'WAR': 0.20385033379909953},
 {'loss': 1.9598106145858765,
  'UAR': 0.1910589452753011,
  'WAR': 0.18319066147859922},
 {'loss': 1.8293211460113525,
  'UAR': 0.17546139250182177,
  'WAR': 0.17229159635504557}]